<a href="https://colab.research.google.com/github/marcplanas11-alt/insurance-claims-triage-ai/blob/main/claims_agentic_workflow_v2_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 13.4 MB/s eta 0:00:00


In [2]:
import anthropic
import json
from getpass import getpass

In [15]:
from google.colab import userdata

# El nombre del secreto no debe tener espacios.
# Asegúrate de renombrarlo en el panel de secretos de Colab.
ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

In [17]:
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Reply with only this JSON: {\"status\": \"ok\"}"
        }
    ]
)

print(response.content[0].text)

```json
{"status": "ok"}
```


In [18]:
claim_text = """
The insured reports water damage to the kitchen caused by a burst pipe.
Estimated repair cost is £8,500.
The incident occurred 18 days ago.
The insured has provided photos but no plumber invoice yet.
"""

policy_text = """
The policy covers sudden and accidental escape of water from fixed domestic installations.
Claims must be notified as soon as reasonably practicable.
Gradual damage, wear and tear, and pre-existing damage are excluded.
The insurer may request invoices, photos, and repair reports.
"""

state = {
    "claim_text": claim_text,
    "policy_text": policy_text,
    "intake": {
        "claim_type": "water_damage",
        "cause": "burst_pipe",
        "estimated_loss": 8500,
        "risk_flags": ["late_notification", "missing_documentation"]
    },
    "coverage_analysis": {
        "coverage_position": "potentially_covered",
        "covered_peril": "escape_of_water",
        "exclusions": ["wear_and_tear", "gradual_damage"],
        "conditions": ["prompt_notification_required"],
        "missing_documents": ["plumber_invoice"],
        "coverage_confidence": 0.75
    },
    "decision": None,
    "confidence": None,
    "risk_flags": ["late_notification", "missing_documentation"]
}

state

{'claim_text': '\nThe insured reports water damage to the kitchen caused by a burst pipe.\nEstimated repair cost is £8,500.\nThe incident occurred 18 days ago.\nThe insured has provided photos but no plumber invoice yet.\n',
 'policy_text': '\nThe policy covers sudden and accidental escape of water from fixed domestic installations.\nClaims must be notified as soon as reasonably practicable.\nGradual damage, wear and tear, and pre-existing damage are excluded.\nThe insurer may request invoices, photos, and repair reports.\n',
 'intake': {'claim_type': 'water_damage',
  'cause': 'burst_pipe',
  'estimated_loss': 8500,
  'risk_flags': ['late_notification', 'missing_documentation']},
 'coverage_analysis': {'coverage_position': 'potentially_covered',
  'covered_peril': 'escape_of_water',
  'exclusions': ['wear_and_tear', 'gradual_damage'],
  'conditions': ['prompt_notification_required'],
  'missing_documents': ['plumber_invoice'],
  'coverage_confidence': 0.75},
 'decision': None,
 'confi

In [19]:
def decision_agent_claude(client, state):
    prompt = f"""
You are an insurance claims decision agent.

You must decide one of:
- APPROVE
- ESCALATE
- REJECT

Return ONLY valid JSON with this exact schema:

{{
  "decision": "APPROVE | ESCALATE | REJECT",
  "confidence": 0.0,
  "risk_flags": [],
  "human_review_required": true,
  "reason": ""
}}

CLAIM:
{state["claim_text"]}

POLICY:
{state["policy_text"]}

INTAKE:
{json.dumps(state["intake"], indent=2)}

COVERAGE_ANALYSIS:
{json.dumps(state["coverage_analysis"], indent=2)}

CURRENT_RISK_FLAGS:
{json.dumps(state["risk_flags"], indent=2)}
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw_output = response.content[0].text
    decision = json.loads(raw_output)

    state["decision"] = decision
    state["confidence"] = decision["confidence"]
    state["risk_flags"] = sorted(list(set(state["risk_flags"] + decision["risk_flags"])))

    return state

In [20]:
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=500,
    messages=[
        {
            "role": "user",
            "content": "Return ONLY valid JSON: {\"status\": \"ok\"}"
        }
    ]
)

raw_output = response.content[0].text
print(repr(raw_output))

'```json\n{"status": "ok"}\n```'


In [21]:
import json

def decision_agent_claude(client, state):
    prompt = f"""
You are an insurance claims decision agent.

You must decide one of:
- APPROVE
- ESCALATE
- REJECT

Return ONLY valid JSON with this exact schema:

{{
  "decision": "APPROVE | ESCALATE | REJECT",
  "confidence": 0.0,
  "risk_flags": [],
  "human_review_required": true,
  "reason": ""
}}

CLAIM:
{state["claim_text"]}

POLICY:
{state["policy_text"]}

INTAKE:
{json.dumps(state["intake"], indent=2)}

COVERAGE_ANALYSIS:
{json.dumps(state["coverage_analysis"], indent=2)}

CURRENT_RISK_FLAGS:
{json.dumps(state["risk_flags"], indent=2)}
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw_output = response.content[0].text
    # Extract JSON string by finding the first '{' and last '}'
    start_index = raw_output.find('{')
    end_index = raw_output.rfind('}')

    if start_index != -1 and end_index != -1 and start_index < end_index:
        json_string = raw_output[start_index : end_index + 1]
    else:
        # Fallback if the curly braces are not found as expected, try stripping common markdown fences.
        json_string = raw_output.strip()
        if json_string.startswith("```json"):
            json_string = json_string[len("```json"):]
        elif json_string.startswith("```"):
            json_string = json_string[len("```"):]
        if json_string.endswith("```"):
            json_string = json_string[:-len("```")]
        json_string = json_string.strip() # Remove any remaining whitespace/newlines

    decision = json.loads(json_string)

    state["decision"] = decision
    state["confidence"] = decision["confidence"]
    state["risk_flags"] = sorted(list(set(state["risk_flags"] + decision["risk_flags"])))

    return state

state = decision_agent_claude(client, state)

state["decision"]

{'decision': 'ESCALATE',
 'confidence': 0.65,
 'risk_flags': ['late_notification',
  'missing_documentation',
  'notification_delay_18_days'],
 'human_review_required': True,
 'reason': "Claim requires human review due to notification delay of 18 days, which may breach the 'as soon as reasonably practicable' policy condition. While the peril (burst pipe) appears covered and photos have been provided, the plumber invoice is still missing, preventing verification that damage was sudden rather than gradual/wear-and-tear. The estimated loss of £8,500 is material. Recommend escalation to assess: (1) whether notification delay was reasonable given circumstances, (2) obtain plumber report to confirm sudden burst vs gradual leak, and (3) verify repair costs with proper documentation before approval."}

In [22]:
print(repr(raw_output))

'```json\n{"status": "ok"}\n```'


In [31]:
import json
import re

def safe_json_parse(raw_output, default_factory=None):
    # Intentar extraer JSON de bloques de código markdown
    json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
    if json_match:
        clean_input = json_match.group(0)
    else:
        clean_input = raw_output

    try:
        return json.loads(clean_input)
    except json.JSONDecodeError:
        if default_factory:
            return default_factory(raw_output)
        return {"error": "invalid_json", "raw": raw_output}

In [24]:
bad_output = "Here is my answer: ESCALATE because documents are missing"

safe_json_parse(bad_output)

{'decision': 'ESCALATE',
 'confidence': 0.0,
 'risk_flags': ['invalid_json_output'],
 'human_review_required': True,
 'reason': 'Model did not return valid JSON. Raw output: Here is my answer: ESCALATE because documents are missing'}

In [25]:
decision = safe_json_parse(raw_output)

In [26]:
state = decision_agent_claude(client, state)
state["decision"]

{'decision': 'ESCALATE',
 'confidence': 0.65,
 'risk_flags': ['late_notification',
  'missing_documentation',
  'notification_delay_18_days',
  'no_professional_assessment'],
 'human_review_required': True,
 'reason': "Claim requires human review due to 18-day notification delay which may breach 'as soon as reasonably practicable' condition, and missing plumber invoice needed to verify sudden vs gradual damage. The peril (burst pipe) is potentially covered under escape of water, but delay raises questions about whether damage was truly sudden/accidental or potentially gradual/pre-existing. Photos provided are positive, but professional plumber assessment is essential to exclude wear and tear. Estimated loss of £8,500 warrants verification before approval. Recommend requesting plumber invoice and explanation for notification delay before final decision."}

In [35]:
def validation_agent_claude(client, state):
    prompt = f"""
You are a claims validation agent.
Your job is to challenge the decision.
Return ONLY valid JSON:
{{
  \"validation_status\": \"PASS | REVIEW_REQUIRED | FAIL\",
  \"issues\": [],
  \"confidence_adjustment\": 0.0,
  \"human_review_required\": true,
  \"reason\": \"\"
}}

DECISION:
{json.dumps(state.get('decision', {}), indent=2)}
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )

    def validation_fallback(raw):
        return {
            "validation_status": "REVIEW_REQUIRED",
            "issues": ["JSON_PARSE_ERROR"],
            "confidence_adjustment": -0.1,
            "human_review_required": True,
            "reason": "Failed to parse model output"
        }

    raw_output = response.content[0].text
    validation = safe_json_parse(raw_output, validation_fallback)

    state["validation"] = validation

    # Use .get() to avoid KeyError if the model omits a field
    adj = validation.get("confidence_adjustment", 0)
    current_conf = state.get("confidence", 0) if state.get("confidence") is not None else 0
    state["confidence"] = max(0, round(current_conf + adj, 2))

    if validation.get("human_review_required"):
        state["human_review_required"] = True

    return state

In [36]:
state = validation_agent_claude(client, state)
state["validation"], state["confidence"]

({'validation_status': 'PASS',
  'issues': [],
  'confidence_adjustment': 0.0,
  'human_review_required': True,
  'reason': "Decision to ESCALATE is appropriate and well-justified. The 18-day notification delay is a legitimate policy concern that could breach 'as soon as reasonably practicable' requirements. The identified risk flags are relevant: late notification creates ambiguity about whether damage was sudden or gradual, missing plumber documentation prevents verification of the cause, and lack of professional assessment makes it difficult to exclude wear and tear or pre-existing conditions. The £8,500 claim value justifies thorough verification. The escalation reasoning is sound, proportionate to the risk level, and the requested additional documentation (plumber invoice and delay explanation) are reasonable requirements to make an informed decision. Confidence level of 0.65 appropriately reflects the uncertainty."},
 0.65)

In [37]:
from typing import TypedDict, Optional, Literal

class ClaimState(TypedDict):

    # --- INPUT ---
    claim_id: str
    claim_text: str

    # --- DECISION ---
    decision: Optional[Literal["APPROVE", "REJECT", "ESCALATE"]]
    confidence: Optional[float]

    # --- VALIDATION ---
    is_valid: Optional[bool]
    validation_errors: Optional[str]

    # --- ROUTING ---
    routing: Optional[Literal["AUTO_PAY", "MANUAL_REVIEW", "FRAUD_CHECK"]]

    # --- GOVERNANCE ---
    human_review_required: bool
    audit_log: list

    # --- CONTROL ---
    error: Optional[str]